# 2000-Sample Prepared-Dataset VAE Workflow

This notebook restores a prepared haptic dataset from Google Drive, builds a reproducible 2000-sample subset, writes a runtime YAML config under `/content`, and runs `train -> reconstruction eval -> PCA -> sweep/labeling -> extended validation` without editing repo configs.


In [ ]:
# 1. Clone repo
import os

REPO_URL = "https://github.com/cindy-77jiayi/thesis_hapticAE.git"
REPO_DIR = "/content/thesis_hapticAE"
BRANCH = "codex/wavcaps-vae-cleanup"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch --all
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}
else:
    !git clone {REPO_URL} {REPO_DIR}
    !git -C {REPO_DIR} checkout {BRANCH}

os.chdir(REPO_DIR)
!git log --oneline -3
print(f"Working directory: {os.getcwd()}")


In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt huggingface_hub hf_transfer


In [ ]:
# 3. Experiment configuration
from pathlib import Path

# Drive / archive paths
PREPARED_ARCHIVE_DRIVE = "/content/drive/MyDrive/thesis_wavcaps_as/prepared_dataset/wavcaps_haptic_prepared_as.tar"
DRIVE_EXPORT_ROOT = "/content/drive/MyDrive/thesis_wavcaps_as/experiments"

# Subset controls
RUN_NAME = "vae_subset2000"
SUBSET_SIZE = 2000
SUBSET_SEED = 42
TRAIN_SPLIT = 0.8
REBUILD_SUBSET = False

# Data / sampling
SR = 8000
T = 80000
ANALYSIS_BATCH_SIZE = 4
TRAIN_RANDOM_SEEK = True
TRAIN_SAMPLE_WITH_REPLACEMENT = False
VAL_RANDOM_SEEK = False
VAL_SAMPLE_WITH_REPLACEMENT = False

# Model
LATENT_DIM = 20
CHANNELS = [32, 64, 128, 128]
FIRST_KERNEL = 25
KERNEL_SIZE = 9
ACTIVATION = "leaky_relu"
NORM = "group"
LOGVAR_CLIP = [-10.0, 10.0]

# Training
BATCH_SIZE = 4
EPOCHS = 80
PATIENCE = 20
EARLY_STOP_START = 10
MIN_DELTA = 1e-4
GRAD_CLIP = 1.0
PRINT_EVERY = 5
LR = 2e-4
WEIGHT_DECAY = 1e-5
SCHED_FACTOR = 0.5
SCHED_PATIENCE = 10

# Loss
RECON_TIME_WEIGHT = 1.8
L1_WEIGHT = 0.2
STFT_LOSS_WEIGHT = 1.0
AMPLITUDE_WEIGHT = 0.7
CLAMP_RANGE = 3.0
STFT_SCALES = [128, 256, 512, 1024]
STFT_HOP_LENGTHS = [32, 64, 128, 256]
STFT_WIN_LENGTHS = [128, 256, 512, 1024]
STFT_SCALE_WEIGHTS = [1.0, 1.0, 0.8, 0.6]
STFT_LINEAR_WEIGHT = 0.02
STFT_LOG_WEIGHT = 0.04
STFT_EPS = 1e-7

# KL
FREE_BITS = 0.5
BETA_MAX = 0.003
N_CYCLES = 2
KL_RATIO = 0.5

# PCA / controls
N_COMPONENTS = 8
BUILD_SWEEP_STEPS = 11
VALIDATE_SWEEP_STEPS = 21

# Runtime paths
PREPARED_DATA_DIR = "/content/data/wavcaps_haptic_prepared_as"
SUBSET_ROOT = f"/content/data/subsets/{RUN_NAME}"
SUBSET_DATA_DIR = f"{SUBSET_ROOT}/data"
SUBSET_META_DIR = f"{SUBSET_ROOT}/metadata"
RUNTIME_CONFIG_PATH = f"/content/runtime_configs/{RUN_NAME}.yaml"
OUTPUT_ROOT = "/content/outputs"
CHECKPOINT_PATH = f"{OUTPUT_ROOT}/{RUN_NAME}/best_model.pt"
EVAL_DIR = f"{OUTPUT_ROOT}/eval_{RUN_NAME}"
PCA_DIR = f"{OUTPUT_ROOT}/pca_{RUN_NAME}"
CTRL_DIR = f"{OUTPUT_ROOT}/controls_{RUN_NAME}"
VAL_DIR = f"{OUTPUT_ROOT}/validation_{RUN_NAME}"

Path("/content/data").mkdir(parents=True, exist_ok=True)
Path("/content/runtime_configs").mkdir(parents=True, exist_ok=True)
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

print("RUN_NAME:", RUN_NAME)
print("PREPARED_ARCHIVE_DRIVE:", PREPARED_ARCHIVE_DRIVE)
print("PREPARED_DATA_DIR:", PREPARED_DATA_DIR)
print("SUBSET_DATA_DIR:", SUBSET_DATA_DIR)
print("RUNTIME_CONFIG_PATH:", RUNTIME_CONFIG_PATH)


In [ ]:
# 4. Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# 5. Restore prepared dataset from Google Drive into /content/data
import subprocess
from pathlib import Path

prepared_root = Path(PREPARED_DATA_DIR)
archive_path = Path(PREPARED_ARCHIVE_DRIVE)
manifest_path = prepared_root / "manifest.jsonl"

assert archive_path.exists(), f"Missing prepared dataset archive: {archive_path}"

if not prepared_root.exists():
    subprocess.run(["tar", "-xf", str(archive_path), "-C", "/content/data"], check=True)
else:
    print(f"Prepared dataset already exists: {prepared_root}")

assert prepared_root.exists(), f"Restore failed: {prepared_root}"
assert manifest_path.exists(), f"Missing manifest: {manifest_path}"

!du -sh {PREPARED_DATA_DIR}
!find {PREPARED_DATA_DIR} -name "*.wav" | wc -l
!wc -l {manifest_path}


In [ ]:
# 6. Build a reproducible 2000-sample subset with symlinks and saved train/val lists
import json
import os
import random
import shutil
from pathlib import Path

prepared_root = Path(PREPARED_DATA_DIR)
subset_root = Path(SUBSET_ROOT)
subset_data_dir = Path(SUBSET_DATA_DIR)
subset_meta_dir = Path(SUBSET_META_DIR)

if subset_root.exists() and REBUILD_SUBSET:
    shutil.rmtree(subset_root)

if not subset_root.exists():
    subset_data_dir.mkdir(parents=True, exist_ok=True)
    subset_meta_dir.mkdir(parents=True, exist_ok=True)

    all_wavs = sorted(prepared_root.rglob("*.wav"))
    assert len(all_wavs) >= SUBSET_SIZE, f"Need at least {SUBSET_SIZE} wavs, found {len(all_wavs)}"

    rng = random.Random(SUBSET_SEED)
    selected = rng.sample(all_wavs, SUBSET_SIZE)
    rng.shuffle(selected)

    train_count = int(len(selected) * TRAIN_SPLIT)
    train_sources = selected[:train_count]
    val_sources = selected[train_count:]
    split_map = {src: "train" for src in train_sources}
    split_map.update({src: "val" for src in val_sources})

    subset_manifest = []
    train_lines = []
    val_lines = []

    for src_wav in sorted(selected):
        rel_wav = src_wav.relative_to(prepared_root)
        src_meta = src_wav.with_suffix(".am1.json")
        dst_wav = subset_data_dir / rel_wav
        dst_meta = subset_data_dir / rel_wav.with_suffix(".am1.json")
        dst_wav.parent.mkdir(parents=True, exist_ok=True)

        if not dst_wav.exists():
            os.symlink(src_wav, dst_wav)
        if src_meta.exists() and not dst_meta.exists():
            os.symlink(src_meta, dst_meta)

        split = split_map[src_wav]
        entry = {
            "split": split,
            "source_wav": str(src_wav),
            "source_meta": str(src_meta) if src_meta.exists() else None,
            "subset_wav": str(dst_wav),
            "subset_meta": str(dst_meta) if dst_meta.exists() else None,
        }
        subset_manifest.append(entry)
        if split == "train":
            train_lines.append(str(dst_wav))
        else:
            val_lines.append(str(dst_wav))

    subset_manifest_path = subset_meta_dir / "subset_manifest.jsonl"
    train_list_path = subset_meta_dir / "train_list.txt"
    val_list_path = subset_meta_dir / "val_list.txt"

    with subset_manifest_path.open("w", encoding="utf-8") as fp:
        for entry in subset_manifest:
            fp.write(json.dumps(entry, ensure_ascii=False) + "\n")

    train_list_path.write_text("\n".join(sorted(train_lines)) + "\n", encoding="utf-8")
    val_list_path.write_text("\n".join(sorted(val_lines)) + "\n", encoding="utf-8")
else:
    print(f"Reusing existing subset at: {subset_root}")

subset_manifest_path = Path(SUBSET_META_DIR) / "subset_manifest.jsonl"
train_list_path = Path(SUBSET_META_DIR) / "train_list.txt"
val_list_path = Path(SUBSET_META_DIR) / "val_list.txt"

assert subset_manifest_path.exists(), subset_manifest_path
assert train_list_path.exists(), train_list_path
assert val_list_path.exists(), val_list_path

n_subset = sum(1 for _ in subset_manifest_path.open("r", encoding="utf-8"))
n_train = sum(1 for line in train_list_path.read_text(encoding="utf-8").splitlines() if line.strip())
n_val = sum(1 for line in val_list_path.read_text(encoding="utf-8").splitlines() if line.strip())

print("subset root:", subset_root)
print("subset samples:", n_subset)
print("train samples:", n_train)
print("val samples:", n_val)
assert n_subset == SUBSET_SIZE, (n_subset, SUBSET_SIZE)
assert n_train == int(SUBSET_SIZE * TRAIN_SPLIT), (n_train, int(SUBSET_SIZE * TRAIN_SPLIT))
assert n_train + n_val == n_subset, (n_train, n_val, n_subset)


In [ ]:
# 7. Write runtime YAML config for existing CLI scripts
import yaml
from pathlib import Path

runtime_cfg = {
    "seed": SUBSET_SEED,
    "model_type": "vae",
    "run_name": RUN_NAME,
    "data": {
        "sr": SR,
        "T": T,
        "scale": 1.0,
        "extensions": [".wav", ".flac"],
        "segment_mode": "hapticgen",
        "normalize_mode": "none",
        "min_segment_ratio": 1.0,
        "clip_range": None,
        "use_minmax": False,
        "train_split": TRAIN_SPLIT,
        "analysis_batch_size": ANALYSIS_BATCH_SIZE,
        "train_random_seek": TRAIN_RANDOM_SEEK,
        "train_sample_with_replacement": TRAIN_SAMPLE_WITH_REPLACEMENT,
        "val_random_seek": VAL_RANDOM_SEEK,
        "val_sample_with_replacement": VAL_SAMPLE_WITH_REPLACEMENT,
        "train_file_list": str(Path(SUBSET_META_DIR) / "train_list.txt"),
        "val_file_list": str(Path(SUBSET_META_DIR) / "val_list.txt"),
    },
    "model": {
        "latent_dim": LATENT_DIM,
        "channels": CHANNELS,
        "first_kernel": FIRST_KERNEL,
        "kernel_size": KERNEL_SIZE,
        "activation": ACTIVATION,
        "norm": NORM,
        "logvar_clip": LOGVAR_CLIP,
    },
    "training": {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "min_delta": MIN_DELTA,
        "early_stop_start": EARLY_STOP_START,
        "grad_clip": GRAD_CLIP,
        "print_every": PRINT_EVERY,
    },
    "optimizer": {
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
    },
    "scheduler": {
        "factor": SCHED_FACTOR,
        "patience": SCHED_PATIENCE,
    },
    "loss": {
        "l1_weight": L1_WEIGHT,
        "stft_loss_weight": STFT_LOSS_WEIGHT,
        "amplitude_weight": AMPLITUDE_WEIGHT,
        "clamp_range": CLAMP_RANGE,
        "recon_time_weight": RECON_TIME_WEIGHT,
        "stft_scales": STFT_SCALES,
        "stft_hop_lengths": STFT_HOP_LENGTHS,
        "stft_win_lengths": STFT_WIN_LENGTHS,
        "stft_scale_weights": STFT_SCALE_WEIGHTS,
        "stft_linear_weight": STFT_LINEAR_WEIGHT,
        "stft_log_weight": STFT_LOG_WEIGHT,
        "stft_eps": STFT_EPS,
    },
    "kl": {
        "free_bits": FREE_BITS,
        "beta_max": BETA_MAX,
        "n_cycles": N_CYCLES,
        "ratio": KL_RATIO,
    },
}

runtime_cfg_path = Path(RUNTIME_CONFIG_PATH)
runtime_cfg_path.parent.mkdir(parents=True, exist_ok=True)
runtime_cfg_path.write_text(yaml.safe_dump(runtime_cfg, sort_keys=False), encoding="utf-8")

print(runtime_cfg_path.read_text(encoding="utf-8"))


In [ ]:
# 8. Subset sanity check
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

subset_wavs = sorted(Path(SUBSET_DATA_DIR).rglob("*.wav"))
print("subset wav count:", len(subset_wavs))

sample_wavs = random.sample(subset_wavs, min(4, len(subset_wavs)))
fig, axes = plt.subplots(len(sample_wavs), 1, figsize=(14, 3 * len(sample_wavs)))
if len(sample_wavs) == 1:
    axes = [axes]

for ax, wav_path in zip(axes, sample_wavs):
    y, sr = sf.read(wav_path)
    ax.plot(y, linewidth=0.7)
    ax.set_title(wav_path.name)
    ax.set_xlim(0, len(y))
    ax.grid(alpha=0.2)
    meta_path = wav_path.with_suffix(".am1.json")
    if meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        print(f"{wav_path.name}: {meta.get('user_prompt', '')}")
        print("  filter_metrics:", meta.get("filter_metrics"))

plt.tight_layout()
plt.show()

stats_subset = random.sample(subset_wavs, min(200, len(subset_wavs)))
mean_abs_vals = []
std_vals = []
length_vals = []
for wav_path in stats_subset:
    y, sr = sf.read(wav_path)
    y = np.asarray(y, dtype=np.float32)
    mean_abs_vals.append(float(np.mean(np.abs(y))))
    std_vals.append(float(np.std(y)))
    length_vals.append(len(y))

print("mean_abs median:", float(np.median(mean_abs_vals)))
print("std median:", float(np.median(std_vals)))
print("length median:", float(np.median(length_vals)))


In [ ]:
# 9. Train VAE on the 2000-sample subset
!python -u scripts/train.py --config "$RUNTIME_CONFIG_PATH" --data_dir "$SUBSET_DATA_DIR" --output_dir "$OUTPUT_ROOT"


In [ ]:
# 10. Reconstruction evaluation
!python scripts/eval.py --config "$RUNTIME_CONFIG_PATH" --data_dir "$SUBSET_DATA_DIR" --checkpoint "$CHECKPOINT_PATH" --output_dir "$EVAL_DIR"


In [ ]:
# 11. Inspect reconstruction quality and loss curves
import json
import numpy as np
import os
import sys
import torch

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.utils.config import load_config
from src.utils.seed import set_seed
from src.data.loaders import build_dataloaders, build_model, load_checkpoint
from src.eval.evaluate import evaluate_reconstruction, print_metrics
from src.eval.visualize import plot_loss_curves, plot_waveform_comparison

config = load_config(RUNTIME_CONFIG_PATH)
set_seed(config.get("seed", 42))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
metrics_path = os.path.join(OUTPUT_ROOT, RUN_NAME, "metrics.npz")

metrics = np.load(metrics_path)
plot_loss_curves(metrics["train_losses"].tolist(), metrics["val_losses"].tolist())

data = build_dataloaders(config, SUBSET_DATA_DIR, batch_size=4)
model = build_model(config, device)
load_checkpoint(model, CHECKPOINT_PATH, device)
result = evaluate_reconstruction(model, data["val_loader"], device, n_samples=4)
print_metrics(result)
plot_waveform_comparison(result["x_np"], result["xhat_np"])

std_ratios = [r["recon_std"] / (r["orig_std"] + 1e-8) for r in result["per_sample"]]
print(f"Mean STD ratio: {np.mean(std_ratios):.2%}")


In [ ]:
# 12. Extract latents and fit PCA
!python scripts/extract_and_pca.py --config "$RUNTIME_CONFIG_PATH" --data_dir "$SUBSET_DATA_DIR" --checkpoint "$CHECKPOINT_PATH" --output_dir "$PCA_DIR" --n_components $N_COMPONENTS


In [ ]:
# 13. PCA coverage
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Z = np.load(f"{PCA_DIR}/Z.npy")
Z_scaled = StandardScaler().fit_transform(Z)
full_pca = PCA().fit(Z_scaled)
cumvar = np.cumsum(full_pca.explained_variance_ratio_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.bar(range(1, len(full_pca.explained_variance_ratio_) + 1), full_pca.explained_variance_ratio_)
ax1.axvline(x=N_COMPONENTS + 0.5, color="r", linestyle="--", label=f"{N_COMPONENTS} PCs")
ax1.set_title("Scree Plot")
ax1.set_xlabel("PC")
ax1.set_ylabel("Explained Variance Ratio")
ax1.legend()

ax2.plot(range(1, len(cumvar) + 1), cumvar, "o-")
ax2.axvline(x=N_COMPONENTS, color="g", linestyle="--", label=f"{N_COMPONENTS} PCs")
ax2.axhline(y=0.9, color="r", linestyle="--", label="90%")
ax2.set_title("Cumulative Coverage")
ax2.set_xlabel("Number of PCs")
ax2.set_ylabel("Coverage")
ax2.legend()
plt.tight_layout()
plt.show()

print(f"PC coverage @{N_COMPONENTS}: {cumvar[N_COMPONENTS - 1]:.2%}")


In [ ]:
# 14. Build controls, sweep gallery, and initial labeling
!python scripts/build_controls.py --config "$RUNTIME_CONFIG_PATH" --data_dir "$SUBSET_DATA_DIR" --checkpoint "$CHECKPOINT_PATH" --output_dir "$CTRL_DIR" --pca_dir "$PCA_DIR" --n_components $N_COMPONENTS --n_sweep_steps $BUILD_SWEEP_STEPS


In [ ]:
# 15. Extended validation
!python scripts/validate_extended.py --config "$RUNTIME_CONFIG_PATH" --data_dir "$SUBSET_DATA_DIR" --checkpoint "$CHECKPOINT_PATH" --output_dir "$VAL_DIR" --controls_dir "$CTRL_DIR" --pca_dir "$PCA_DIR" --n_components $N_COMPONENTS --n_sweep_steps $VALIDATE_SWEEP_STEPS


In [ ]:
# 16. Review outputs
import json
from IPython.display import Markdown, Image, display

with open(f"{CTRL_DIR}/controls_spec.json", "r", encoding="utf-8") as f:
    spec = json.load(f)

print(f"n_controls: {spec['n_controls']}")
print(f"total explained variance: {spec['total_explained_variance_pct']:.1f}%")
display(Markdown(open(f"{CTRL_DIR}/controls_table.md", "r", encoding="utf-8").read()))
display(Image(filename=f"{CTRL_DIR}/metric_trends.png", width=1000))
display(Image(filename=f"{VAL_DIR}/monotonicity_extended_heatmap.png", width=1000))
display(Image(filename=f"{VAL_DIR}/selectivity_extended.png", width=800))


In [ ]:
# 17. Save results, runtime config, and subset metadata back to Google Drive
import json
import shutil
import subprocess
import time
from pathlib import Path

export_root = Path(DRIVE_EXPORT_ROOT) / f"{RUN_NAME}_{time.strftime('%Y%m%d_%H%M%S')}"
export_root.mkdir(parents=True, exist_ok=True)

def copy_if_exists(src, dst_dir):
    src = Path(src)
    dst_dir = Path(dst_dir)
    if not src.exists():
        print(f"skip missing: {src}")
        return
    dst_dir.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        shutil.copytree(src, dst_dir / src.name, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst_dir / src.name)
    print(f"saved: {src}")

copy_if_exists(Path(OUTPUT_ROOT) / RUN_NAME, export_root / "outputs")
copy_if_exists(EVAL_DIR, export_root / "outputs")
copy_if_exists(PCA_DIR, export_root / "outputs")
copy_if_exists(CTRL_DIR, export_root / "outputs")
copy_if_exists(VAL_DIR, export_root / "outputs")
copy_if_exists(RUNTIME_CONFIG_PATH, export_root / "runtime")
copy_if_exists(Path(SUBSET_META_DIR) / "subset_manifest.jsonl", export_root / "subset")
copy_if_exists(Path(SUBSET_META_DIR) / "train_list.txt", export_root / "subset")
copy_if_exists(Path(SUBSET_META_DIR) / "val_list.txt", export_root / "subset")

summary = {
    "run_name": RUN_NAME,
    "subset_size": SUBSET_SIZE,
    "subset_seed": SUBSET_SEED,
    "train_split": TRAIN_SPLIT,
    "runtime_config": RUNTIME_CONFIG_PATH,
    "subset_data_dir": SUBSET_DATA_DIR,
    "checkpoint": CHECKPOINT_PATH,
}
with open(export_root / "save_summary.json", "w", encoding="utf-8") as fp:
    json.dump(summary, fp, ensure_ascii=False, indent=2)

print("Saved experiment to:", export_root)
